# 06 — Error Analysis

Uses **actual** held-out test predictions from `reports/tables/test_predictions.csv`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import project_path

pred_path = project_path("reports/tables/test_predictions.csv")
assert pred_path.exists(), "Run scripts/train_evaluate.py first"
df = pd.read_csv(pred_path, parse_dates=["timestamp"])
df["error_type"] = "correct"
df.loc[(df.y_pred == 1) & (df.y_episode == 0), "error_type"] = "false_positive"
df.loc[(df.y_pred == 0) & (df.y_episode == 1), "error_type"] = "false_negative"
display(df.error_type.value_counts())
display(df.groupby("error_type")[["PM2.5", "y_prob"]].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df, x="y_prob", hue="error_type", element="step", ax=ax)
ax.set_title("Predicted probability by error type (test)")
plt.tight_layout()
plt.savefig(project_path("reports/figures/error_prob_by_type.png"), dpi=150)
plt.show()

hc = df[((df.y_prob >= 0.8) & (df.y_pred != df.y_episode)) | ((df.y_prob <= 0.2) & (df.y_pred != df.y_episode))]
print("high-confidence errors:", len(hc))
hc.head(20)